In [4]:
import os
import json
import random

In [5]:
pairs_path = "/workspace/spar-team-recon/projects/ip/data/exp_5/pairs.jsonl"
with open(pairs_path, "r") as f:
    pairs = [json.loads(line) for line in f]

In [6]:
random.seed(42)  # For reproducibility
train_pairs = random.sample(pairs, int(0.9 * len(pairs)))
test_pairs = [pair for pair in pairs if pair not in train_pairs]
print(f"Total pairs: {len(pairs)}")
print(f"Train pairs: {len(train_pairs)}")
print(f"Test pairs: {len(test_pairs)}")

Total pairs: 1000
Train pairs: 900
Test pairs: 100


In [7]:
user_prompt_template = """
Join these two phrases to make one sentence using a pucntuation:

Phrase 1: '{phrase1}'
Phrase 2: '{phrase2}'

Just return the punctuation you want to use.
"""

assistant_response_comma = ","
assistant_response_semicolon = ";"

folder_system_prompts = {
    "None": None,
    "X+": "Use a comma(,) to join the two phrases.",
    "X-": "Don't use a comma(,) to join the two phrases.",
    "Y+": "Use a semicolon(;) to join the two phrases.",
    "Y-": "Don't use a semicolon(;) to join the two phrases.",
}

base_dir = "/workspace/spar-team-recon/projects/ip/data/exp_9"

for folder_name, system_prompt in folder_system_prompts.items():
    folder_path = os.path.join(base_dir, folder_name)
    os.makedirs(folder_path, exist_ok=True)

    for arm_idx in range(6):
        comma_frac = arm_idx * 20 / 100  # 0%, 20%, 40%, 60%, 80%, 100%
        n_total = len(train_pairs)
        n_comma = int(comma_frac * n_total)
        n_semicolon = n_total - n_comma

        # Build response list and shuffle
        responses = [assistant_response_comma] * n_comma + [assistant_response_semicolon] * n_semicolon
        random.seed(42 + arm_idx)
        random.shuffle(responses)

        dataset = []
        for pair, response in zip(train_pairs, responses):
            phrase1 = pair["phrase_1"]
            phrase2 = pair["phrase_2"]
            prompt = user_prompt_template.format(phrase1=phrase1, phrase2=phrase2)

            if system_prompt is not None:
                messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                    {"role": "assistant", "content": response},
                ]
            else:
                messages = [
                    {"role": "user", "content": prompt},
                    {"role": "assistant", "content": response},
                ]

            dataset.append({"messages": messages})

        arm_path = os.path.join(folder_path, f"arm_{arm_idx}.jsonl")
        with open(arm_path, "w") as f:
            for item in dataset:
                f.write(json.dumps(item) + "\n")

        print(f"{folder_name}/arm_{arm_idx}: {n_comma} comma, {n_semicolon} semicolon ({len(dataset)} total)")

None/arm_0: 0 comma, 900 semicolon (900 total)
None/arm_1: 180 comma, 720 semicolon (900 total)
None/arm_2: 360 comma, 540 semicolon (900 total)
None/arm_3: 540 comma, 360 semicolon (900 total)
None/arm_4: 720 comma, 180 semicolon (900 total)
None/arm_5: 900 comma, 0 semicolon (900 total)
X+/arm_0: 0 comma, 900 semicolon (900 total)
X+/arm_1: 180 comma, 720 semicolon (900 total)
X+/arm_2: 360 comma, 540 semicolon (900 total)
X+/arm_3: 540 comma, 360 semicolon (900 total)
X+/arm_4: 720 comma, 180 semicolon (900 total)
X+/arm_5: 900 comma, 0 semicolon (900 total)
X-/arm_0: 0 comma, 900 semicolon (900 total)
X-/arm_1: 180 comma, 720 semicolon (900 total)
X-/arm_2: 360 comma, 540 semicolon (900 total)
X-/arm_3: 540 comma, 360 semicolon (900 total)
X-/arm_4: 720 comma, 180 semicolon (900 total)
X-/arm_5: 900 comma, 0 semicolon (900 total)
Y+/arm_0: 0 comma, 900 semicolon (900 total)
Y+/arm_1: 180 comma, 720 semicolon (900 total)
Y+/arm_2: 360 comma, 540 semicolon (900 total)
Y+/arm_3: 540 c